

## 1. One-Hot Encoding

* **Concept:** Converts each unique category into a new binary column ($0$ or $1$).
* **Best Used For:** Nominal categorical data (categories with **no natural order** or ranking, e.g., colors, cities).
* **Why:** Prevents distance-based models from incorrectly assuming that category $2$ is "greater" or "better" than category $1$.

### Example

$$\text{Original Data} \rightarrow \begin{bmatrix} \text{Red} \\ \text{Blue} \\ \text{Green} \end{bmatrix}$$

$$\text{One-Hot Encoded} \rightarrow \begin{array}{ccc} \text{Color\_Red} & \text{Color\_Blue} & \text{Color\_Green} \\ 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{array}$$

---

## 2. Label Encoding (Ordinal Encoding)

* **Concept:** Assigns a unique integer ($0, 1, 2, \dots$) to each distinct category.
* **Best Used For:** Ordinal categorical data (categories that have a **clear hierarchy or order**), or target variables ($y$) in classification tasks.
* **Caution:** Avoid using this on non-ordered features with linear models or distance-based algorithms, as $3 > 1$ implies a non-existent magnitude.

### Example

$$\begin{array}{lcc} \text{Education Level} & \rightarrow & \text{Encoded Value} \\ \hline \text{High School} & \rightarrow & 0 \\ \text{Bachelor's} & \rightarrow & 1 \\ \text{Master's} & \rightarrow & 2 \\ \text{PhD} & \rightarrow & 3 \end{array}$$

---

## 3. Target Encoding (Mean Encoding)

* **Concept:** Replaces a categorical value with the **mean of the target variable ($y$)** for that specific category.
* **Best Used For:** High-cardinality features (features with hundreds or thousands of unique categories, like zip codes) where One-Hot Encoding would create too many columns.
* **Caution:** Highly prone to **overfitting/data leakage** if computed across the entire dataset without cross-validation!

### Example

Suppose we want to predict **House Price ($y$)** based on **Neighborhood ($X$)**:

$$\begin{array}{lc} \text{Neighborhood} & \text{Price } (y) \\ \hline \text{Downtown} & \$500\text{k} \\ \text{Downtown} & \$600\text{k} \\ \text{Suburbs} & \$200\text{k} \\ \text{Suburbs} & \$300\text{k} \end{array}$$

1. Calculate average price per neighborhood:
* **Downtown Mean:** $(500 + 600) / 2 = \$550\text{k}$
* **Suburbs Mean:** $(200 + 300) / 2 = \$250\text{k}$


2. Replace categories with their target means:

$$\begin{array}{lc} \text{Neighborhood (Encoded)} \\ \hline 550,000 \\ 550,000 \\ 250,000 \\ 250,000 \end{array}$$

---

## 4. K-Fold Target Encoding

* **Concept:** To prevent target encoding from leaking target values into the training set, encoding is performed **inside a K-Fold cross-validation scheme**.
* **How it works:**
1. Split the dataset into $K$ folds.
2. For Fold 1, calculate target means using **only Folds 2 to $K$** (the training folds).
3. Use those calculated means to encode the data in **Fold 1** (the validation fold).
4. Repeat for all $K$ folds.



### Example (3-Fold)

Given 6 rows of data divided into 3 folds:

$$\begin{array}{ccllc} \text{Fold} & \text{Category} & \text{Target } (y) \\ \hline 1 & A & 1 & \leftarrow \text{To encode this fold...} \\ 1 & B & 0 & \leftarrow \text{...we calculate target means using Folds 2 \& 3 ONLY.} \\ 2 & A & 1 \\ 2 & B & 1 \\ 3 & A & 0 \\ 3 & B & 0 \end{array}$$

* Target mean for **A** in Folds 2 & 3: $(1 + 0) / 2 = 0.5$
* Target mean for **B** in Folds 2 & 3: $(1 + 0) / 2 = 0.5$
* Fold 1 rows get encoded as $A = 0.5$ and $B = 0.5$, completely protecting Fold 1's actual target values from leaking into its own encodings.

---

## 5. Leave-One-Out (LOO) Target Encoding

* **Concept:** A variation of target encoding where each row's encoded value is calculated using the target mean of all **other** rows in the dataset, **excluding the current row's target**.
* **Formula for row $i$:**

$$\text{Encoded}_i = \frac{\left(\sum \text{Target}_{\text{category}}\right) - y_i}{N_{\text{category}} - 1}$$

* **Why:** It removes the current row's target value from its own feature encoding, preventing direct self-leakage while avoiding full K-fold complexity.

### Example

Suppose we have 4 rows for Category **"X"**:

$$\begin{array}{ccc} \text{Row} & \text{Category} & \text{Target } (y) \\ \hline 1 & X & 1 \\ 2 & X & 1 \\ 3 & X & 0 \\ 4 & X & 0 \end{array}$$

* Total Target Sum for **X** = $1 + 1 + 0 + 0 = 2$
* Total Count ($N$) = $4$

Now encode each row individually:

* **Row 1:** $\frac{2 - 1}{4 - 1} = \frac{1}{3} \approx 0.33$
* **Row 2:** $\frac{2 - 1}{4 - 1} = \frac{1}{3} \approx 0.33$
* **Row 3:** $\frac{2 - 0}{4 - 1} = \frac{2}{3} \approx 0.67$
* **Row 4:** $\frac{2 - 0}{4 - 1} = \frac{2}{3} \approx 0.67$

Notice how rows with a target of $1$ receive a slightly lower encoding ($0.33$) than rows with a target of $0$ ($0.67$). To prevent the model from exploiting this inverted relationship, noise is often added to LOO encodings in practice.